# Test de la fonction d'association de station à des échangeurs

In [1]:
from datetime import date
from pathlib import Path

from enum import StrEnum
import folium
import json
import geopandas as gpd
import pandas as pd

CHUNK_SIZE: int = 20 #0
SAMPLES: int = 288  # 5 min
SATURE_H: int = 45  # minimum duration (min) of saturation to have a saturated hour
MAX_SESSION_DURATION_HOURS = 24

ID_POC: str = "id_pdc_itinerance"
ID_STATION: str = "id_station_itinerance"
ID_POOL: str = "id_pool"
SATURATION_RATIO = 0.1
OVERLOAD_RATIO = 0.2
MIN_POWER = 75
RAYON_ECHANGEUR = 3000
BUFFER_ECHANGEUR = 100

#day = date(2026,7, 5)
#day = date(2026,7, 14)
day = date(2026,7, 25)
#day = date(2026,8, 1)
#day = date(2026,8, 18)
#day = date(2026,8, 25)
date_file = f"{day.year}{day.month:02d}{day.day:02d}"

data_quali = "../data/"

class NaturePool(StrEnum):
    AIRE = "Aire"
    ECHANGEUR = "Echangeur"
    ALL = "All"

In [2]:
def read_statics(day: date, min_power: float) -> pd.DataFrame:
    """Read static data for pocs and stations."""
    date_statics = f"{day.day:02d}-{day.month:02d}-{day.year}"
    e5_str = pd.read_csv(f"../saturation/data_DMR_e2_e3/e5_{date_statics}.csv")["extras"][0]
    statics = pd.DataFrame(json.loads(e5_str))
    statics["unite"] = statics["id_pdc_itinerance"].str[:5]
    return statics[statics["puissance_nominale"] >= min_power]

def read_statics_pools(day:date, min_power: float) -> pd.DataFrame:
    """Read static data for stations and pools."""
    e1_statics = pd.read_csv("../saturation/source/tests_DMR/aires_pdc_2026-07-25.csv")[[ID_POOL, ID_STATION]].drop_duplicates()
    return e1_statics


In [3]:
def stations_echangeurs(
    stations: gpd.GeoDataFrame,
    echangeurs: gpd.GeoDataFrame,
) -> gpd.GeoDataFrame:
    """Return the nearest echangeur for each station within a maximum distance."""
    stations = stations.drop_duplicates(subset=[ID_STATION])
    ech_geo = gpd.GeoDataFrame(echangeurs, geometry=gpd.points_from_xy(echangeurs["longitude"], echangeurs["latitude"]), crs="EPSG:4326")
    stations_geo = gpd.GeoDataFrame(
        stations, 
        geometry=gpd.points_from_xy(stations["longitude"], stations["latitude"]), 
        crs="EPSG:4326")
   
    ech_station = gpd.sjoin_nearest(
        stations_geo[[ID_STATION, "geometry"]].to_crs(epsg=2154),
        ech_geo[[ID_POOL, "geometry"]].to_crs(epsg=2154),
        how="inner",
        max_distance=RAYON_ECHANGEUR,
        distance_col="distance",
    )
    return ech_station[[ID_STATION, "geometry", ID_POOL, "distance"]].reset_index(drop=True).to_crs(epsg=4326)

In [4]:
def surfaces_echangeurs(
    points_stations: gpd.GeoDataFrame,
    echangeurs: gpd.GeoDataFrame,
) -> gpd.GeoDataFrame:
    """Return the convex hull surfaces of stations and echangeurs for each pool."""

    points = pd.concat([points_stations[[ID_POOL, "geometry"]], echangeurs[[ID_POOL, "geometry"]]], ignore_index=True)

    surfaces = points.to_crs(epsg=2154).dissolve(by=ID_POOL)
    surfaces["geometry"] = surfaces.geometry.convex_hull.buffer(BUFFER_ECHANGEUR)
  
    return surfaces.to_crs(epsg=4326).reset_index()

In [5]:
def init_pools(file_path: Path, filter: NaturePool = NaturePool.AIRE) -> gpd.GeoDataFrame:
    """Initialize pools from a geojson file."""
    #pools = pd.read_csv(pools_file).rename(columns={"id_aire": "id_pool"})
    pools = gpd.read_file(file_path).rename(columns={"id_aire": "id_pool"})
    match filter:
        case NaturePool.AIRE:
            return pools[pools[ID_POOL].str[:1] == "A"]
        case NaturePool.ECHANGEUR:
            return pools[pools[ID_POOL].str[:1] == "E"]
        case _:
            return pools

In [6]:
dmr_file = "../saturation/data_DMR/aires_dmr_2026-08-03.geojson"
echangeurs = init_pools(dmr_file, NaturePool.ECHANGEUR)

c:\Users\phili\anaconda3\envs\afir_V12_20240806\Lib\site-packages\pyogrio\core.py:26: RuntimeWarning: Could not detect GDAL data files.  Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


In [7]:
min_power = 50
statics = read_statics(day, min_power)
statics = statics[statics["puissance_nominale"] >= min_power]


In [8]:
ech_station = stations_echangeurs(statics, echangeurs)
ech_station

,id_station_itinerance,geometry,id_pool,distance
0,FREVCP000379,POINT (2.78232 48.85336),E000034,2365.691914
1,FREVZP28EE7D709C7C4189B1B9336CC95C7,POINT (1.38459 44.03711),E000376,620.571566
2,FRTSLP413867,POINT (1.38441 44.00557),E000377,148.515913
3,FRIZFPFAST738,POINT (1.18454 49.24927),E000308,377.429495
4,FRIOYP13530974,POINT (-0.49106 47.5003),E000281,302.319842
...,...,...,...,...
1857,FRECHPCSUTC,POINT (4.83675 46.81995),E000116,1864.589166
1858,FRECHPCFFUV,POINT (0.7014 47.42555),E000232,2448.131515
1859,FRSIGPSIGE236,POINT (2.55802 48.84872),E000027,1498.653938
1860,FRLDLPLFR4325EVCP,POINT (4.71908 45.9424),E000613,393.621223


In [9]:
surf = surfaces_echangeurs(ech_station, echangeurs)
surf

,id_pool,geometry
0,E000001,"POLYGON ((2.51621 48.98923, 2.51609 48.98918, ..."
1,E000002,"POLYGON ((2.54184 49.03385, 2.54184 49.03377, ..."
2,E000003,"POLYGON ((2.55434 49.07163, 2.55432 49.07155, ..."
3,E000004,"POLYGON ((2.56367 49.13352, 2.56367 49.13344, ..."
4,E000005,"POLYGON ((2.6122 49.20618, 2.61228 49.2061, 2...."
...,...,...
1081,E001082,"POLYGON ((6.76636 45.91557, 6.76635 45.91548, ..."
1082,E001083,"POLYGON ((6.76292 45.92582, 6.76291 45.92573, ..."
1083,E001084,"POLYGON ((6.74962 45.92733, 6.74961 45.92724, ..."
1084,E001085,"POLYGON ((1.79542 42.57962, 1.79541 42.57953, ..."


In [10]:
import webbrowser
from pathlib import Path

In [11]:
refnat = {'location': [46.3, 2.3], 'zoom_start': 7}
#refnat  = {'tiles':'CartoDB positron', 'location': [46.3, 2.3], 'zoom_start': 7}
carte = surf.explore(m=folium.Map(**refnat), color='green', name='aires', show=True)
carte.save("aires.html")
webbrowser.open(Path("aires.html").resolve().as_uri())

True

In [12]:
import folium

m = folium.Map(
    location=[46.6, 2.5],
    zoom_start=6,
    tiles="CartoDB positron",
)

m

In [13]:
import sys

print(sys.executable)

c:\Users\phili\anaconda3\envs\afir_V12_20240806\python.exe
